In [ ]:
import os
os.chdir("..")

In [ ]:
from st_repl import SpatialReg
import pandas as pd
import geopandas as gpd
import polars as pl
from pathlib import Path
import tempfile
from jp_qcew import CleanQCEW
from spreg import dgp_lag
from CensusForge import CensusAPI
import statsmodels.api as sm
from libpysal import weights
import numpy as np

from jp_tools import download
sr = SpatialReg()
capi = CensusAPI()
cq = CleanQCEW("data/")

In [ ]:
sr.spatial_data(alpha=0.5, beta=0.5, sigma=2,rho=0.7, seed=787)

In [ ]:
year = 2019
qtr = 3

rng_global = np.random.default_rng(seed=seed + (year * 10 + qtr))
# Number of observations

slice_df = gdf[
    (gdf["year"] == year) & (gdf["qtr"] == qtr)
].reset_index()
X = slice_df[["total_employment", "total_wages"]].values.reshape(-1, 2)
X = sm.add_constant(X)

n_obs = len(gdf)

# Define Beta coefficients
coef = np.array([200, alpha, beta])

# Compute XB matrix
xb = X @ coef
xb = xb.reshape(-1, 1)

u = rng_global.normal(loc=0, scale=sigma, size=n_obs).reshape(-1, 1)

# calculate the spatial lag
y_true = dgp_lag(u, xb, self.wq, rho=rho)

slice_df["y_true"] = y_true
slice_df["centroid"] = slice_df.centroid
slice_df["lat"] = slice_df["centroid"].x
slice_df["lon"] = slice_df["centroid"].y
slice_df["w_rook"] = weights.lag_spatial(self.wr, y_true)


In [ ]:
gdf = sr.quasi_data()
gdf.centroid

In [ ]:
gdf["log"] = np.log(gdf["total_wages"])
gdf

In [ ]:
gdf[(gdf["year"] == 2018) &(gdf["qtr"] == 3)]


In [ ]:
X = gdf[["total_employment", "total_wages"]].values.reshape(-1, 2)
X = sm.add_constant(X)
X

In [ ]:
n_obs = len(gdf)

# Create the independent variables for 4 variables that com from a normal distribution X ~ N(mu, sigma)
X = np.ones((n_obs, 4))

for i in range(1, 4):
    rng = np.random.default_rng(seed=787 + i)
    X[:, i] = rng.normal(loc=5, scale=2, size=n_obs)

In [ ]:
X